# Introduction 

MongoDB est une base de données open source codée en C++ basée sur un concept de stockage sous la forme de documents au format JSON. Le grand avantage de ce système est l'optimisation de la mémoire. Dans une base relationelle, chaque colonne doit être définie au préalable avec une empreinte mémoire et un type de donnée. Dans une base MongoDB si le champ n'est pas présent, il n'apparait pas dans un document et n'impacte pas la mémoire, alors qu'en SQL la place mémoire est utilisée même si le champ est absent, pour spécifier que la valeur est null.

## Les avantages 

- Optimisé pour le multi-machines et la réplication de données
- Pas besoin de jointures entre les tables compte tenu du modèle de données sous forme de documents
- Un index est créé sur chaque clé pour une rapidité de requêtes
- Un langage de requêtage aussi puissant que le SQL
- Optimisation de la mémoire
- Pas besoin de définir un schéma de données à l'avance

## Inconvenients

- Malgré ses optimisations Mongo a tendance à consommer plus de stockage qu'une BDD relationnelle
- Les Queries et le Aggrégations peuvent être plus compliqué surtout lorsque des requêtes neccessite des données de plusieurs collections
- Pas de schéma "enforcement" ce qui veut dire que des documents dans une même collection peuvent avoir des données différents et inattendu

## Mongo VS BDD relationnelle

Le choix d’une base de données dépend du besoin auquel on essaie de répondre.

En général, Mongo est choisi comme BDD lorsqu’on a affaire à des données non structurées ou semi-structurées, ainsi que dans des environnements de données dynamiques où il est difficile de prévoir à l’avance un schéma structuré.

Quant aux BDD relationnelles (MySQL, PostgreSQL, …), elles sont meilleures pour stocker des données structurées, avoir une gestion de données cohérente et permettre de requêter un gros volume de données rapidement et de manière optimale.


## Concepts basiques

- Database : une database est un regroupement de collections. Chaque database possède son propre système de fichiers et sa propre authentification. Elle a le même rôle qu'une database en MySQL.
- Collection : une collection est un regroupement de documents. C'est une table en MySQL, la principale différence étant qu'elle ne définit pas un schéma de données fixe. Les documents présents dans une collection n'ont pas forcément tous les mêmes champs.
- Document : un document est un objet JSON stocké sous la forme de plusieurs paires clé:valeur. C'est l'équivalent d'une ligne dans une table SQL.
- Champ : Un champ est l'équivalent d'une colonne en SQL. Il permet de faire des requêtes.

## Identifiants

Tous les documents possèdent un identifiant unique, ce qui permet de le retrouver très efficacement. L'identifiant peut être spécifié lors de l'ajout d'un nouveau document (nom+prenom, adresse email, url, etc). Dans le cas ou aucun identifiant n'est précisé, MongoDB se charge d'en ajouter un. Il est composé d'un nombre stocké sur 12 bytes au format hexadécimal :

- Les 4 premiers bytes sont le timestamp de l'ajout du document
- Les 3 suivants correspondent à l'identifiant de la machine
- Les 2 suivants l'identifiant du processus
- Les 3 derniers sont une valeur incrémentale


## Les types de données

Une base de données MongoDB permet de stocker un grand volume de données hétérogènes sans imposer un modèle de données fixe pour tous les documents. Il est conseillé, comme vu plus haut, de bien définir la structure globale pour garder une cohérence tout au long des développements.

- Integer : entier relatif stocké sur 32 ou 64 bits.
- Double : nombre décimal stocké sur 64 bits.
- String : chaine de caractère (encodée en utf-8)
- Booléen : True ou False
- Object : sous-objet stocké au format JSON
- Date : date au format UNIX (nombre de ms écoulées depuis le 1er janvier 1970) stockée sur 64 bits.
- Array : stocker une liste d'éléments au format atomique ou d'objets

D'autres types sont disponibles et vous pouvez les trouver https://docs.mongodb.com/manual/reference/bson-types/

# Installation

L'installation peut se faire de plusieurs manières.

Par exemple, directement depuis les sources ou à partir de packages, notamment sur des machines tournant sur des distributions Linux comme Debian.  
Liens vers le tutorial https://docs.mongodb.com/manual/tutorial/install-mongodb-on-ubuntu/

Ce n'est pas cette méthode que nous allons utiliser. Comme vous avez pu le voir au début du cours, nous pouvons utiliser docker pour faire tourner des services dans des containers isolés sur notre machine.


L'avantage de Docker est qu'il n'installe aucune dépendance sur votre machine et laisse son environnement propre. Lien vers le tutorial : https://hub.docker.com/_/mongo/

Lancez Mongo dans un container docker :
```bash
> docker run --name my-mongo -d -p 27017:27017 mongo
```

On peut voir les containers qui tournent actuellement avec la commande
```bash
> docker ps
```

On peut regarder aussi regarder les logs avec :
```bash
> docker logs my-mongo
```

Pour rappel, un des avantages de Docker est sa portabilité, ainsi il est possible de faire tourner un mongo de la même manière peut importe la machine !

# Connexion

Pour se connecter à une base Mongo deux solutions sont possibles. Dans les deux cas, la syntaxe Mongo est utilisée pour effectuer des requêtes.

En ligne de commande avec Mongo Shell (mongosh). Il suffit de l'installer depuis ce lien : https://www.mongodb.com/docs/mongodb-shell/install/

 Ensuite d'executer la commande :
 ```bash
> mongosh "mongodb://localhost:27017"
```

Vous vous retouvrez ainsi dans une session mongosh connectée à votre base local tournant sur Docker.

L'autre solution est d'utiliser un client graphique mongo qui permet de visuellement gérer sa base de données. Il existe beaucoup de client mongo, parmi les plus connus :
- MongoDB Compass le client officiel => https://www.mongodb.com/products/tools/compass
- Robo3T un client open source => https://robomongo.org/
- Des plugins d'IDE qui permettent directement dans votre IDE favori de gérer votre mongo, VS code for mongo, DataGrip, ... 

Une fois que vous avez installé un client il suffit de renseigner la string de connexion pour vous connecter à votre base

N'hésitez pas à tester les deux solutions et choisir celle que vous préférez. Il est généralement intéressant de commencer par mongosh pour bien comprendre ce qu'il se passe sous le capot.

# Création d'un modèle de données

La création d'un modèle de données clair et adapté est une tâche importante et primordiale. Ce modèle de données doit être réfléchi à court et long terme et doit prendre en compte la capacité de stockage et les besoins métiers.

## Database


A partir du shell Mongo, on peut afficher les databases disponibles. Au démarrage, aucune n'est créée à part celles par défaut:
```
> show dbs
admin   0.000GB
config  0.000GB
local   0.000GB
```

On peut créer une database de test
```
> use test
switched to db test
```

Pour supprimer définitivement une database:

```
> db.dropDatabase()
```
Comme vous pouvez le deviner cette commande est à utiliser avec précaution.

## Collection

Les collections correspondent aux tables en SQL. Elles sont des sous-ensembles d'une database. Pour créer une collection il faut auparavant s'être référencé sur une database.

```
show dbs
use <YOUR_DB_NAME>
db.createCollection(<YOUR_COLLECTION_NAME>)
show collections
```

Comme pour les databases on peut vouloir supprimer définitivement une collection.

```
db.<YOUR_COLLECTION_NAME>.drop()
show collections
```


## Document

Un document (objet JSON) est un sous-ensemble d'une collection qui est lui même une sous-partie d'une database. Pour insérer un document il faut donc se référencer sur une database et sur la collection souhaitée.

```
use <YOUR_DB_NAME>
db.createCollection(<YOUR_COLLECTION_NAME>)
show collections
db.<YOUR_COLLECTION_NAME>.insert({
    firstname : "Thomas",
    lastname : "Shelby",
    position : "director",
    company : "Peaky Blinders"})
```

Si vous ne précisez pas d'identifiant unique (id du document), MongoDB se charge de le remplir avec les règles définies précédement. Une bonne pratique est de trouver une règle permettant de retrouver facilement et efficacement un document sans avoir à faire une requête complexe et obliger la base à rechercher dans ses champs. Une technique est de prendre le hash d'une combinaison des champs qui permet de créer une clé unique SHA128(firstname+lastname+position) par exemple.

```
use <YOUR_DB_NAME>
show collections
db.<YOUR_COLLECTION_NAME>.insert({
    firstname : "Thomas",
    lastname : "Shelby",
    position : "CEO",
    gender : "Male",
    age : 35,
    description : "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
    nicknames : ["Tom", "Tommy", "Thomas"],
    company : "Peaky Blinders",
    episodes : [1,2,4,5,6]
    })
```
Pour des soucis de performances, si un grand nombre de documents doivent être insérés très rapidement sans surcharger les appels réseaux, il est possible de passer une liste JSON d'objets à la fonction insert.

```
db.<YOUR_COLLECTION_NAME>.insert([
{
    firstname : "Arthur",
    lastname : "Shelby",
    position : "Associate",
    gender : "Male",
    age : 38,
    description : "Arthur Shelby Jr. is the eldest of the Shelby siblings and the tough member of Peaky Blinders, the Deputy Vice President Shelby Company Limited. He's also a member of the ICA.",
    company : "Peaky Blinders",
    episodes : [1,4,6]

},{
    firstname : "John",
    lastname : "Shelby",
    position : "Associate",
    gender : "Male",
    age : 30,
    description : "John Michael Shelby, also called Johnny or John Boy, was the third of Shelby siblings and a member of the Peaky Blinders.",
    nicknames : ["Johnny", "John Boy"],
    company : "Peaky Blinders",
    episodes : [4,5,6]
},{
    firstname : "Ada",
    lastname : "Thorne",
    position : "HR",
    gender : "Female",
    age : 28,
    description : "Ada Thorne is the fourth and only female of the Shelby sibling. She's the Head of Acquisitions of the Shelby Company Limited.",
    nicknames : ["Ada Shelby"],
    company : "Peaky Blinders",
    episodes : [1,2,6]
},{
    firstname : "Michael",
    lastname : "Gray",
    position : "Accounting",
    gender : "Male",
    age : 21,
    description : "Michael Gray is the son of Polly Shelby, his father is dead, and cousin of the Shelby siblings. He is the Chief Accountant in the Shelby Company Limited.",
    nicknames : ["Henry Johnson", "Jobbie Muncher", "Mickey"],
    company : "Peaky Blinders",
    episodes : [5,6]
},{
    firstname : "Polly",
    lastname : "Gray",
    gender : "Female",
    age : 45,
    position : "CFO",
    description : "Elizabeth Polly Gray (née Shelby) is the matriarch of the Shelby Family, aunt of the Shelby siblings, the treasurer of the Birmingham criminal gang, the Peaky Blinders, a certified accountant and company treasurer of Shelby Company Limited. ",
    nicknames : ["Aunt Polly", "Polly Gray", "Elizabeth Gray", "Polly Shelby", "Pol"],
    company : "Peaky Blinders",
    episodes : [1,2,5,6]
}])
```

Vous pouvez voir tous les documents de votre collection avec : 
```bash
db.<YOUR_COLLECTION_NAME>.find()
```

# Python 

Il existe une API Python développée pour interagir avec une base de données MongoDB. 

Ce package s'appelle `pymongo`, vous pouvez trouver la documentation https://www.mongodb.com/docs/languages/python/pymongo-driver/current/. Il est important d'avoir des APIs dans les différents langages pour faciliter l'intégration dans les applications.

le package est déjà installé dans votre virtual env mais si jamais:

```bash
pipenv install pymongo==4.9.1
```


Ce package garde très largement la syntaxe Mongo shell et permet d'utiliser ces méthodes et items (DataBases, Collections, Documents) en tant qu'objets Python.

Dans un premier temps, on veut créer le lien entre notre base Mongo et notre programme, pour cela on créer un `client`.

In [1]:
from pymongo import MongoClient

client = MongoClient()

Permet de se connecter à une base MongoDB en créant un pointeur client vers cette base. Par défault ce client est paramétré sur le localhost et le port 27017 qui est le port par défaut et très généralement utilisé.
 
Néanmoins, vous pouvez choisir de vous connecter à n'importe quelle base, qu'elle soit sur votre machine ou à distance.


```
client = MongoClient("http://<YOUR_IP_ADDRESS>:<YOUR_PORT_NUMBER>)
```

Il est possible comme depuis le MongoShell de lister les bases de données.


In [2]:
print(client.list_database_names())

['admin', 'config', 'local']



Vous pouvez aussi les sélectionner. Il y a deux syntaxes : 
```
db = client.<YOUR_DATABASE_NAME>
```
ou 
```
db = client["<YOUR_DATABASE_NAME>"]
```


In [3]:
db_series = client.series
print(type(db_series))

<class 'pymongo.synchronous.database.Database'>


Si la database ou la collections que vous selectionnez via ces commandes n'existe pas, elle est alors créée.

In [4]:
collection_peaky = db_series['peaky'] 

In [5]:
collection_peaky.insert_one(
    {
    "firstname" : "Thomas",
    "lastname" : "Shelby",
    "position" : "CEO",
    "gender" : "Male",
    "age" : 35,
    "description" : "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
    "nicknames" : ["Tom", "Tommy", "Thomas"],
    "company" : "Peaky Blinders",
    "episodes" : [1,2,4,5,6]
    })

InsertOneResult(ObjectId('693823c9ae8f36e44f888385'), acknowledged=True)

In [6]:
DOCUMENTS = [
    {
    "firstname" : "Thomas",
    "lastname" : "Shelby",
    "position" : "CEO",
    "gender" : "Male",
    "age" : 35,
    "description" : "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
    "nicknames" : ["Tom", "Tommy", "Thomas"],
    "company" : "Peaky Blinders",
    "episodes" : [1,2,4,5,6]
    }, 
{
    "firstname" : "Arthur",
    "lastname" : "Shelby",
    "position" : "Associate",
    "gender" : "Male",
    "age" : 38,
    "description" : "Arthur Shelby Jr. is the eldest of the Shelby siblings and the tough member of Peaky Blinders, the Deputy Vice President Shelby Company Limited. He's also a member of the ICA.",
    "company" : "Peaky Blinders",
    "episodes" : [1,4,6]

},{
    "firstname" : "John",
    "lastname" : "Shelby",
    "position" : "Associate",
    "gender" : "Male",
    "age" : 30,
    "description" : "John Michael Shelby, also called Johnny or John Boy, was the third of Shelby siblings and a member of the Peaky Blinders.",
    "nicknames" : ["Johnny", "John Boy"],
    "company" : "Peaky Blinders",
    "episodes" : [4,5,6]
},{
    "firstname" : "Ada",
    "lastname" : "Thorne",
    "position" : "HR",
    "gender" : "Female",
    "age" : 28,
    "description" : "Ada Thorne is the fourth and only female of the Shelby sibling. She's the Head of Acquisitions of the Shelby Company Limited.",
    "nicknames" : ["Ada Shelby"],
    "company" : "Peaky Blinders",
    "episodes" : [1,2,6]
},{
    "firstname" : "Michael",
    "lastname" : "Gray",
    "position" : "Accounting",
    "gender" : "Male",
    "age" : 21,
    "description" : "Michael Gray is the son of Polly Shelby, his father is dead, and cousin of the Shelby siblings. He is the Chief Accountant in the Shelby Company Limited.",
    "nicknames" : ["Henry Johnson", "Jobbie Muncher", "Mickey"],
    "company" : "Peaky Blinders",
    "episodes" : [5,6]
},{
    "firstname" : "Polly",
    "lastname" : "Gray",
    "gender" : "Female",
    "age" : 45,
    "position" : "CFO",
    "description" : "Elizabeth Polly Gray (née Shelby) is the matriarch of the Shelby Family, aunt of the Shelby siblings, the treasurer of the Birmingham criminal gang, the Peaky Blinders, a certified accountant and company treasurer of Shelby Company Limited. ",
    "nicknames" : ["Aunt Polly", "Polly Gray", "Elizabeth Gray", "Polly Shelby", "Pol"],
    "company" : "Peaky Blinders",
    "episodes" : [1,2,5,6]
}]

In [7]:
collection_peaky.insert_many(DOCUMENTS)

InsertManyResult([ObjectId('693823d9ae8f36e44f888386'), ObjectId('693823d9ae8f36e44f888387'), ObjectId('693823d9ae8f36e44f888388'), ObjectId('693823d9ae8f36e44f888389'), ObjectId('693823d9ae8f36e44f88838a'), ObjectId('693823d9ae8f36e44f88838b')], acknowledged=True)

In [8]:
db_series.list_collection_names()

['peaky']

La base de données a bien été créée. Nous allons voir comment faire des requêtes

## Requêter

Afin de récupérer les documents stockés dans une collection, des fonctions de requête sont disponibles. La fonction `find()` permet de récupérer les N premiers documents. La fonction `find_one()` permet de récupérer le premier élément.

In [9]:
collection_peaky.find_one()

{'_id': ObjectId('693823c9ae8f36e44f888385'),
 'firstname': 'Thomas',
 'lastname': 'Shelby',
 'position': 'CEO',
 'gender': 'Male',
 'age': 35,
 'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
 'nicknames': ['Tom', 'Tommy', 'Thomas'],
 'company': 'Peaky Blinders',
 'episodes': [1, 2, 4, 5, 6]}

Cette fonction permet de récupérer le premier élément de la collection sous la forme d'un dictionnaire.


C'est un peu différent pour la méthode find(). Cela crée, pour des raisons de performances, un curseur PyMongo. En effet, les données seront récupérées uniquement si elles sont utilisées. C'est intéressant pour des collections très volumineuses.

In [10]:
cursor = collection_peaky.find()
type(cursor)

pymongo.synchronous.cursor.Cursor

Un curseur est un type d'iterateur python, pour récupérer l'élément suivant:

In [12]:
next(cursor)


{'_id': ObjectId('693823d9ae8f36e44f888386'),
 'firstname': 'Thomas',
 'lastname': 'Shelby',
 'position': 'CEO',
 'gender': 'Male',
 'age': 35,
 'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
 'nicknames': ['Tom', 'Tommy', 'Thomas'],
 'company': 'Peaky Blinders',
 'episodes': [1, 2, 4, 5, 6]}

In [13]:
for document in cursor :
    print('-----')
    print(document)

-----
{'_id': ObjectId('693823d9ae8f36e44f888387'), 'firstname': 'Arthur', 'lastname': 'Shelby', 'position': 'Associate', 'gender': 'Male', 'age': 38, 'description': "Arthur Shelby Jr. is the eldest of the Shelby siblings and the tough member of Peaky Blinders, the Deputy Vice President Shelby Company Limited. He's also a member of the ICA.", 'company': 'Peaky Blinders', 'episodes': [1, 4, 6]}
-----
{'_id': ObjectId('693823d9ae8f36e44f888388'), 'firstname': 'John', 'lastname': 'Shelby', 'position': 'Associate', 'gender': 'Male', 'age': 30, 'description': 'John Michael Shelby, also called Johnny or John Boy, was the third of Shelby siblings and a member of the Peaky Blinders.', 'nicknames': ['Johnny', 'John Boy'], 'company': 'Peaky Blinders', 'episodes': [4, 5, 6]}
-----
{'_id': ObjectId('693823d9ae8f36e44f888389'), 'firstname': 'Ada', 'lastname': 'Thorne', 'position': 'HR', 'gender': 'Female', 'age': 28, 'description': "Ada Thorne is the fourth and only female of the Shelby sibling. Sh

Il est possible de passer des arguments à la fonction find() ou find_one().

In [14]:
cur = collection_peaky.find({"lastname":"Shelby"})
next(cur)

{'_id': ObjectId('693823c9ae8f36e44f888385'),
 'firstname': 'Thomas',
 'lastname': 'Shelby',
 'position': 'CEO',
 'gender': 'Male',
 'age': 35,
 'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
 'nicknames': ['Tom', 'Tommy', 'Thomas'],
 'company': 'Peaky Blinders',
 'episodes': [1, 2, 4, 5, 6]}


Les différentes opérations mathématiques sont implémentées. 

- Egalité :  `{key:value}` Correspondance clé valeur entre le champ et la requête.
- Différence (not equal) :  {key: {&#36;ne:value}}
- Plus (Grand|Petit) que :  les opérateurs sont &#36;lt (lower than) ; &#36;lte (lower than equals) ; &#36;gt (greater than) ; &#36;gte (greater than equals) : `{key: {<OPERATEUR>:value}}`


In [15]:
cur = collection_peaky.find({"age":{"$gte" :30}})
next(cur)

{'_id': ObjectId('693823c9ae8f36e44f888385'),
 'firstname': 'Thomas',
 'lastname': 'Shelby',
 'position': 'CEO',
 'gender': 'Male',
 'age': 35,
 'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
 'nicknames': ['Tom', 'Tommy', 'Thomas'],
 'company': 'Peaky Blinders',
 'episodes': [1, 2, 4, 5, 6]}

Les opérations logiques sont aussi disponibles.

OR &#36;or et AND &#36;and permettent de faire des requêtes complexes sur une collection. 



In [16]:
cur = collection_peaky.find({"$and":[{"age":{"$gte": 28, "$lt":40}}, {"lastname":"Shelby"}]})
next(cur)

{'_id': ObjectId('693823c9ae8f36e44f888385'),
 'firstname': 'Thomas',
 'lastname': 'Shelby',
 'position': 'CEO',
 'gender': 'Male',
 'age': 35,
 'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
 'nicknames': ['Tom', 'Tommy', 'Thomas'],
 'company': 'Peaky Blinders',
 'episodes': [1, 2, 4, 5, 6]}

### Requêtes complexes

Les objets Mongo peuvent être assez complexes et les requêtes doivent pouvoir matcher tous types de documents.

Pour requêter les valeurs d'une liste : 

In [17]:
cur = collection_peaky.find( { "nicknames":  ["Henry Johnson", "Jobbie Muncher", "Mickey"] } )
next(cur)

{'_id': ObjectId('693823d9ae8f36e44f88838a'),
 'firstname': 'Michael',
 'lastname': 'Gray',
 'position': 'Accounting',
 'gender': 'Male',
 'age': 21,
 'description': 'Michael Gray is the son of Polly Shelby, his father is dead, and cousin of the Shelby siblings. He is the Chief Accountant in the Shelby Company Limited.',
 'nicknames': ['Henry Johnson', 'Jobbie Muncher', 'Mickey'],
 'company': 'Peaky Blinders',
 'episodes': [5, 6]}

Le champ `nicknames` doit matcher exactement la liste donnée en argument (en contenu et en ordre). 


In [19]:
cur = collection_peaky.find( { "nicknames":  ["Henry Johnson",  "Mickey", "Jobbie Muncher"] } )
list(cur)

[]

Lorsqu'aucun document n'a été trouvé le curseur est vide et donc renvoie une erreur lorsque l'on essaye de récupérer le prochain élément.

Si maintenant on veut récupérer tous les documents avec "Mickey" et "Jobbie Muncher", peu importe l'ordre d'apparition et peu importe les autres éléments du tableau.


In [20]:
cur = collection_peaky.find( { "nicknames":  {"$all" :["Mickey", "Jobbie Muncher"] } } )
next(cur)

{'_id': ObjectId('693823d9ae8f36e44f88838a'),
 'firstname': 'Michael',
 'lastname': 'Gray',
 'position': 'Accounting',
 'gender': 'Male',
 'age': 21,
 'description': 'Michael Gray is the son of Polly Shelby, his father is dead, and cousin of the Shelby siblings. He is the Chief Accountant in the Shelby Company Limited.',
 'nicknames': ['Henry Johnson', 'Jobbie Muncher', 'Mickey'],
 'company': 'Peaky Blinders',
 'episodes': [5, 6]}

On peut vouloir maintenant récupérer tous les documents comptenant "Mickey" dans les nicknames (listes). 


In [21]:
cur = collection_peaky.find( { "nicknames": "Mickey" } )
next(cur)

{'_id': ObjectId('693823d9ae8f36e44f88838a'),
 'firstname': 'Michael',
 'lastname': 'Gray',
 'position': 'Accounting',
 'gender': 'Male',
 'age': 21,
 'description': 'Michael Gray is the son of Polly Shelby, his father is dead, and cousin of the Shelby siblings. He is the Chief Accountant in the Shelby Company Limited.',
 'nicknames': ['Henry Johnson', 'Jobbie Muncher', 'Mickey'],
 'company': 'Peaky Blinders',
 'episodes': [5, 6]}

Comme on vient de le voir, une requête sur le champ d'une liste se construit de la même manière qu'une requête sur un champ 'basique'.

La syntaxe générique d'une requête Mongo est la suivante.
```
    db.<YOUR_COLLECTION_NAME>.find( { <array field>: { <operator1>: <value1>, ... } })
```

### Limitation, Projection et Tris

Pour des raisons de performances, il peut être intéressant de limiter les accès réseaux. Pour cela, on peut sélectionner les champs devant être retournés (Projection). On peut aussi demander de limiter le nombre de documents (Limitation).

La syntaxe est la suivante : 

```
db.<YOUR_COLLECTION_NAME>.find(QUERY, PROJECTION).LIMIT(N_DOCUMENTS)
```

Un exemple de projection en utilisant les requêtes déjà utilisées plus haut.


In [22]:
cur = collection_peaky.find({"lastname":"Shelby"}, {"position":1})
next(cur)

{'_id': ObjectId('693823c9ae8f36e44f888385'), 'position': 'CEO'}

Avec une requête plus complexe et une autre projection.

In [23]:
cur = collection_peaky.find({"$and":[{"age":{"$gte": 28, "$lt":40}}, {"lastname":"Shelby"}]}, {"firstname":1})
next(cur)

{'_id': ObjectId('693823c9ae8f36e44f888385'), 'firstname': 'Thomas'}

Un exemple de limitation : 

In [24]:
cur = collection_peaky.find({"lastname":"Shelby"}).limit(2)
list(cur)

[{'_id': ObjectId('693823c9ae8f36e44f888385'),
  'firstname': 'Thomas',
  'lastname': 'Shelby',
  'position': 'CEO',
  'gender': 'Male',
  'age': 35,
  'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
  'nicknames': ['Tom', 'Tommy', 'Thomas'],
  'company': 'Peaky Blinders',
  'episodes': [1, 2, 4, 5, 6]},
 {'_id': ObjectId('693823d9ae8f36e44f888386'),
  'firstname': 'Thomas',
  'lastname': 'Shelby',
  'position': 'CEO',
  'gender': 'Male',
  'age': 35,
  'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
  'nickn

Il est aussi possible de passer directement au Nième document avec la fonction `skip()`.

In [25]:
list(collection_peaky.find({"lastname":"Shelby"}))

[{'_id': ObjectId('693823c9ae8f36e44f888385'),
  'firstname': 'Thomas',
  'lastname': 'Shelby',
  'position': 'CEO',
  'gender': 'Male',
  'age': 35,
  'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
  'nicknames': ['Tom', 'Tommy', 'Thomas'],
  'company': 'Peaky Blinders',
  'episodes': [1, 2, 4, 5, 6]},
 {'_id': ObjectId('693823d9ae8f36e44f888386'),
  'firstname': 'Thomas',
  'lastname': 'Shelby',
  'position': 'CEO',
  'gender': 'Male',
  'age': 35,
  'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
  'nickn

In [26]:
cur = collection_peaky.find({"lastname":"Shelby"}).skip(1)
next(cur)

{'_id': ObjectId('693823d9ae8f36e44f888386'),
 'firstname': 'Thomas',
 'lastname': 'Shelby',
 'position': 'CEO',
 'gender': 'Male',
 'age': 35,
 'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
 'nicknames': ['Tom', 'Tommy', 'Thomas'],
 'company': 'Peaky Blinders',
 'episodes': [1, 2, 4, 5, 6]}

On peut trier les résultats récupérés. 

Pour trier dans l'ordre ascendant et donc récupérer le plus jeune de la famille:

In [27]:
cur = collection_peaky.find({"lastname":"Shelby"}, {"firstname":1}).sort([("age", 1)])
next(cur)

{'_id': ObjectId('693823d9ae8f36e44f888388'), 'firstname': 'John'}

Pour trier dans l'ordre descendant :


In [28]:
cur = collection_peaky.find({"lastname":"Shelby"}, {"firstname":1}).sort([("age", -1)])
next(cur)

{'_id': ObjectId('693823d9ae8f36e44f888387'), 'firstname': 'Arthur'}

On peut aussi trier selon une clé puis une autre.

In [29]:
cur = collection_peaky.find({"lastname":"Shelby"}, {"firstname":1}).sort([("age", -1), ("firstname", 1)])
next(cur)

{'_id': ObjectId('693823d9ae8f36e44f888387'), 'firstname': 'Arthur'}

## Indexation

L'indexation permet d'accélérer les performances sur les requêtes. Si aucun index n'est mis en place, MongoDB doit effectuer un scan de tous les documents pour trouver ceux qui sont pertinents. L'index permet de stocker les valeurs d'un champ de façon triée pour limiter le nombre de document à parcourir pour effectuer une requête. 

Vous pouvez accéder à plus d'informations ici : https://docs.mongodb.com/manual/indexes/

### Indexation simple

L'indexation simple permet de créer l'index en fonction d'un seul champ. 

On spécifie alors l'ordre dans lequel l'index est créé et trié. 

Dans l'ordre croissant, 

In [30]:
collection_peaky.create_index([("age", 1)])

'age_1'

In [31]:
collection_peaky.index_information()

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'age_1': {'v': 2, 'key': [('age', 1)]}}

Et dans l'ordre décroissant, 

In [32]:
collection_peaky.create_index([("age", -1)])

'age_-1'

In [33]:
collection_peaky.index_information()

{'_id_': {'v': 2, 'key': [('_id', 1)]},
 'age_1': {'v': 2, 'key': [('age', 1)]},
 'age_-1': {'v': 2, 'key': [('age', -1)]}}

Pour supprimer un index en particulier : 

In [34]:
collection_peaky.drop_index("age_1")

In [35]:
collection_peaky.index_information()

{'_id_': {'v': 2, 'key': [('_id', 1)]},
 'age_-1': {'v': 2, 'key': [('age', -1)]}}

Pour supprimer tous les index : 

In [36]:
collection_peaky.drop_indexes()

In [37]:
collection_peaky.index_information()

{'_id_': {'v': 2, 'key': [('_id', 1)]}}

Les performances ne sont visibles que pour des collections de taille importante. Mais se ressente très rapidement et sont souvent indispensables. On peut par exemple passer de plusieurs secondes par requêtes à seulement quelques milliseconds ! Ce qui est un gain non négligeable.

### Indexation composée

L'indexation composée permet de créer un index basé sur deux champs différents. L'ordre des champs spécifié dans la création d'un index est important.On peut trier dans l'ordre croissant le premier champ et dans l'ordre décroissant le deuxième champ. 

Il est très utile quand plusieurs champs sont souvent utilisés conjointement pour effectuer des queries. 

In [38]:
collection_peaky.create_index([("age", -1), ("firstname", 1)])

'age_-1_firstname_1'

In [39]:
collection_peaky.index_information()

{'_id_': {'v': 2, 'key': [('_id', 1)]},
 'age_-1_firstname_1': {'v': 2, 'key': [('age', -1), ('firstname', 1)]}}


### Indexations spéciales

Mongo permet plusieurs indexations : 

- Text : permet de faire de la recherche naturelle de *queries* dans du texte. Cet index peut devenir très rapidement très important et prendre beaucoup de place mémoire. Cet index textuel contient un index par mot contenu dans l'ensemble des documents. Il peut aussi être très lent à créer.
- Multiclés : permet de créer un index sur les éléments d'objets stockés dans des listes ou *arrays*.
- 2D, 2DSphere, geoHaystack : permet de créer des index sur des données géospatiales.
- Hash : permet de stocker les valeurs des champs sous forme de *hash*.

Dans ce cours, on se contentera de faire de l'indexation textuelle.

Tous ces mécanismes d'indexation permettent d'accélérer les performances des requêtes. Mais ils peuvent avoir des effets négatifs: 

- Sur l'occupation mémoire : Chaque index doit avoir un minimum de 8 kB et peut prendre beaucoup de place sur le disque et dans la mémoire RAM.
- Sur le temps d'exécution : les opérations d'insertion et d'écriture peuvent être longues puisque Mongo doit insérer chaque nouveau document dans l'index en plus de l'insertion dans la collection.

Exemple : 

Pour créer un index textuel sur la description des personnages : 


In [40]:
collection_peaky.create_index([("description",  "text")])

'description_text'

In [41]:
collection_peaky.index_information()

{'_id_': {'v': 2, 'key': [('_id', 1)]},
 'age_-1_firstname_1': {'v': 2, 'key': [('age', -1), ('firstname', 1)]},
 'description_text': {'v': 2,
  'key': [('_fts', 'text'), ('_ftsx', 1)],
  'weights': SON([('description', 1)]),
  'default_language': 'english',
  'language_override': 'language',
  'textIndexVersion': 3}}

Uniquement après que cet index de texte ait été créé, on peut utiliser la méthode `find()` avec l'argument `$text` pour faire une requête dans le texte.


In [42]:
cur = collection_peaky.find( { "$text": { "$search": "female" } } )
next(cur)

{'_id': ObjectId('693823d9ae8f36e44f888389'),
 'firstname': 'Ada',
 'lastname': 'Thorne',
 'position': 'HR',
 'gender': 'Female',
 'age': 28,
 'description': "Ada Thorne is the fourth and only female of the Shelby sibling. She's the Head of Acquisitions of the Shelby Company Limited.",
 'nicknames': ['Ada Shelby'],
 'company': 'Peaky Blinders',
 'episodes': [1, 2, 6]}

    
#### Exercice

Supprimez tous les index créés et réessayez de faire la même requête textuelle. Que se passe-t-il ?



**Explication de l'exercice:**

L'objectif est de comprendre l'importance des index textuels pour les recherches textuelles en MongoDB.

**Étapes à suivre:**

1. **Supprimez tous les index** avec la commande `drop_indexes()` 
2. **Réessayez la même requête textuelle** avec `$text` et `$search`
3. **Observez le résultat** - Une erreur devrait s'afficher

**Pourquoi cela se produit-il?**

- MongoDB a **besoin d'un index texte** pour effectuer des recherches textuelles avec l'opérateur `$text`
- Sans cet index, MongoDB ne sait pas comment traiter l'opérateur `$search`
- Une `OperationFailure` ou une erreur de requête apparaîtra

**Cela démontre que:**
- Les index ne sont pas optionnels pour certaines opérations
- L'opérateur `$text` dépend entièrement de l'existence d'un index texte préalablement créé
- C'est une bonne pratique de vérifier quels index existent avec `index_information()` avant de les supprimer


In [43]:
# Étape 1 : Vérifier les index existants
print("Index avant suppression:")
print(collection_peaky.index_information())
print("\n" + "="*50 + "\n")

# Étape 2 : Supprimer tous les index
collection_peaky.drop_indexes()
print("Tous les index ont été supprimés!")
print("\nIndex après suppression:")
print(collection_peaky.index_information())
print("\n" + "="*50 + "\n")

# Étape 3 : Essayer la même requête textuelle
try:
    cur = collection_peaky.find( { "$text": { "$search": "female" } } )
    result = next(cur)
    print("Résultat:", result)
except Exception as e:
    print(f"❌ ERREUR: {type(e).__name__}")
    print(f"Message: {e}")

Index avant suppression:
{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'age_-1_firstname_1': {'v': 2, 'key': [('age', -1), ('firstname', 1)]}, 'description_text': {'v': 2, 'key': [('_fts', 'text'), ('_ftsx', 1)], 'weights': SON([('description', 1)]), 'default_language': 'english', 'language_override': 'language', 'textIndexVersion': 3}}


Tous les index ont été supprimés!

Index après suppression:
{'_id_': {'v': 2, 'key': [('_id', 1)]}}


❌ ERREUR: OperationFailure
Message: text index required for $text query, full error: {'ok': 0.0, 'errmsg': 'text index required for $text query', 'code': 27, 'codeName': 'IndexNotFound'}


## Mettre à jour

La mise à jour des documents et une opération très courante dans les bases de données. MongoDB implémente trois fonctions différentes permettant de mettre à jour un ou plusieurs documents à la fois.

- Mettre à jour un seul document : 

```
db.<YOUR_COLLECTION_NAME>.updateOne(<filter>, <update>, <options>)
```

- Le champ `filter` est une requête comme on vient de voir précédemment ; 
- Le champ `update` permet de préciser la requête de mise à jour ;
- Le champ `option` permet de donner des arguments à cette opération.

 Cette fonction va mettre à jour le premier élément renvoyé par la requête `filter`. 
 
 Par exemple : 
 


In [44]:
result = collection_peaky.update_one({"firstname":"Thomas"}, {"$set":{"maincharacter":True}})
result

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

In [45]:
cur = collection_peaky.find_one({"firstname":"Thomas"})
cur

{'_id': ObjectId('693823c9ae8f36e44f888385'),
 'firstname': 'Thomas',
 'lastname': 'Shelby',
 'position': 'CEO',
 'gender': 'Male',
 'age': 35,
 'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
 'nicknames': ['Tom', 'Tommy', 'Thomas'],
 'company': 'Peaky Blinders',
 'episodes': [1, 2, 4, 5, 6],
 'maincharacter': True}

Ici le champ update utilise le selecteur `$set` qui permet de définir les couples clé:valeurs à mettre à jour.

- Mettre à jour une liste de documents : 

```
db.<YOUR_COLLECTION_NAME>.update_many(<filter>, <update>, <options>)
```

Cette fonction va mettre à jour tous les documents concernés par la requête `filter`.
Dans l'exemple ci-dessous nous allons mettre à jour tous les éléments correspondant à la family Shelby.


In [46]:
result = collection_peaky.update_many({"lastname":"Shelby"}, {"$set":{"shelbyFamily":True}})
result

UpdateResult({'n': 4, 'nModified': 4, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

In [47]:
cur = collection_peaky.find({"lastname":"Shelby"})
list(cur)

[{'_id': ObjectId('693823c9ae8f36e44f888385'),
  'firstname': 'Thomas',
  'lastname': 'Shelby',
  'position': 'CEO',
  'gender': 'Male',
  'age': 35,
  'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determined to move his family up in the world.",
  'nicknames': ['Tom', 'Tommy', 'Thomas'],
  'company': 'Peaky Blinders',
  'episodes': [1, 2, 4, 5, 6],
  'maincharacter': True,
  'shelbyFamily': True},
 {'_id': ObjectId('693823d9ae8f36e44f888386'),
  'firstname': 'Thomas',
  'lastname': 'Shelby',
  'position': 'CEO',
  'gender': 'Male',
  'age': 35,
  'description': "Thomas 'Tommy' Michael Shelby M.P. OBE, is the leader of the Birmingham criminal gang Peaky Blinders and the patriarch of the Shelby Family. His experiences during and after the First World War have left him disillusioned and determin


L'option `upsert` peut être très intéressante. Elle permet d'ajouter un document s'il n'existe pas déjà directement depuis la fonction `update()`. Par défaut, cette option est `false`. 
 
 
```
db.<YOUR_COLLECTION_NAME>.update(<filter>, <update>, {upsert: true})
```
    

## Supprimer 

Pour supprimer des documents, comme pour la mise à jour, il existe deux méthodes : 

- `deleteMany({ <field1>: <value1>, ... }`
- `deleteOne({ <field1>: <value1>, ... }`

Pour supprimer tous les documents possédant une condition : 


In [48]:
result = collection_peaky.delete_many({"lastname": "Gray"})
result

DeleteResult({'n': 2, 'ok': 1.0}, acknowledged=True)

In [50]:
cur = collection_peaky.find({"lastname": "Gray"})
list(cur)

[]

Plus aucun document ne possède ces propriétés.

Pour supprimer un seul document (ou le premier si la condition n'est pas assez restrictive) :

In [51]:
result = collection_peaky.delete_one({"firstname": "Arthur"})
result

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

In [53]:
cur = collection_peaky.find({"firstname": "Arthur"})
list(cur)

[]

Pour supprimer tous les documents de la collection : 


In [54]:
result = collection_peaky.delete_many({})
result

DeleteResult({'n': 4, 'ok': 1.0}, acknowledged=True)

In [56]:
cur = collection_peaky.find()
list(cur)

[]

Plus aucun document n'est présent dans la collection

Quelques choses à savoir : 

- La méthode `deleteMany()` applique une fonction à tous les documents. Toutes les fonctions en Mongo sont atomiques ce qui veut dire qu'elles s'appliquent à chaque document indépendamment les uns des autres.
- La méthode `delete()` ne supprime pas les index, même si on supprime tous les documents de la collection.


### Exercice 

Maintenant que la collection est vide, réintégrez les données précédemment supprimés. 

**Solution de l'exercice:**

L'objectif est de réinsérer les documents qui ont été supprimés précédemment. Vous disposez de la liste `DOCUMENTS` définie plus tôt dans le notebook qui contient tous les personnages de Peaky Blinders.

**Étapes:**
1. Réutilisez la liste `DOCUMENTS` définie précédemment
2. Utilisez `insert_many()` pour insérer tous les documents en une seule opération
3. Vérifiez que les données ont bien été réintégrées avec `find()`


In [64]:
# Réinsérer les documents supprimés
collection_peaky.insert_many(DOCUMENTS)
print(f"✓ {len(DOCUMENTS)} documents réinsérés avec succès!")

# Vérification : afficher le nombre de documents dans la collection
count = collection_peaky.count_documents({})
print(f"Nombre total de documents: {count}")

# Afficher les noms des personnages réintégrés
cur = collection_peaky.find({}, {"firstname": 1, "lastname": 1})
print("\nPersonnages réintégrés:")
for doc in cur:
    print(f"  - {doc['firstname']} {doc['lastname']}")

✓ 6 documents réinsérés avec succès!
Nombre total de documents: 6

Personnages réintégrés:
  - Thomas Shelby
  - Arthur Shelby
  - John Shelby
  - Ada Thorne
  - Michael Gray
  - Polly Gray


## Aggregation


Une aggrégation permet de faire des opérations complexes sur des groupes de documents directement dans la base. Elle se charge de grouper les documents entre eux suivant la requête et se charge d'effectuer une opération sur l'ensemble des documents de chacun des groupes. On peut retrouver les mêmes opérations en SQL avec les arguments `GROUP BY`.

La syntaxe est très similaire à toutes les autres fonctions Mongo mais la requête va être plus complexe. 

```
db.<YOUR_COLLECTION_NAME>.aggregate(AGGREGATE_OPERATION)
```

On peut vouloir récupérer le nombre de personnages de chaque famille présente dans la série : 


In [57]:
cur = collection_peaky.aggregate([{"$group" : {"_id" : "$lastname", "charactereNumberByFamily" : {"$sum" : 1}}}])
list(cur)

[]


Vous avez accès à toutes les opérations mathématiques dont vous avez besoin : 

- &#36;sum : fait la somme de 
- &#36;avg : fait la moyenne 
- &#36;min : récupère la valeur minimale 
- &#36;max : récupère la valeur maximale 
- &#36;first : récupère le premier élément
- &#36;last : récupère le dernier élément


In [58]:
cur = collection_peaky.aggregate([{"$group" : {"_id" : "$lastname", "averageAgeByFamily" : {"$avg" : "$age"}}}])
list(cur)


[]

In [59]:
cur = collection_peaky.aggregate([{"$group" : {"_id" : "$lastname", "minAgeByFamily" : {"$min" : "$age"}}}])
list(cur)


[]

In [60]:
cur = collection_peaky.aggregate([{"$group" : {"_id" : "$lastname", "lastAgeByFamily" : {"$last" : "$age"}}}])
list(cur)


[]

On peut ajouter un paramètre à la fonction `aggregate()` pour filtrer les élements à aggréger.
Si on ne veut récupérer que les hommes : 


In [62]:
cur = collection_peaky.aggregate([
        {"$match":{"gender":"Male"}},
        {"$group" : {"_id" : "$lastname", "averageAgeByFamily" : {"$avg" : "$age"}}}
    ])
list(cur)

[]

Il y a peu de temps Mongo supportait des opérations Map Reduce avec du javascript. Depuis les dernières versions le ap reduce est déprécié en faveur des pipelines d'aggregation. Ces pipelines permettent de transformer et manipulé les données complexes de manière flexibles en décrivant les différentes transformations et aggrégations à utiliser dans notre pipeline.

In [63]:
pipeline = [
    {"$match": {"gender": "Male"}},
    {"$group": {"_id": "$lastname", "sumAge": {"$sum": "$age"}}}
]

result = list(collection_peaky.aggregate(pipeline))

for res in result:
    print(res)

Ces méthodes sont surtout utilisées dans un contexte avec une volumétrie de données importantes

N'hésitez pas à explorer Mongosh et un client graphique avant de passer à la suite pour vous familiariser avec l'écosystème.

Deux exercices sont disponibles pour mettre en pratique ce que vous avez appris : `ExerciceKickstarter.ipynb`et `ExerciceYoutube.ipynb`